In [7]:
!pip install monai

In [32]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import nibabel as nib
from sklearn.model_selection import StratifiedGroupKFold
import torch
import monai

import torch
import torch.nn as nn
import monai
import timm
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from pytorch_lightning.loggers import WandbLogger
from sklearn.utils.class_weight import compute_class_weight
import wandb

RANDOM_STATE = 1488

In [9]:
os.environ["WANDB_API_KEY"] = "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"

In [10]:
DATA_DIR = Path("/kaggle/input/competitions/lidc-lung-nodule-classification")
train_path = DATA_DIR / "train.parquet"
test_path = DATA_DIR / "test.csv"
SCANS_DIR = Path("/kaggle/input/datasets/vladimirsydor/lidc-classification-ct-scans") 

In [11]:
train_df = pd.read_parquet(train_path)
test_df = pd.read_csv(test_path)

In [12]:
def image_id_to_path(image_id, scans_dir=SCANS_DIR):
    return scans_dir / f"{image_id}.nii"

In [13]:
train_df["image_path"] = train_df["image_id"].apply(lambda x: image_id_to_path(x))

# Existence check
existing = train_df["image_path"].apply(lambda p: Path(p).exists())
print("Train: rows with existing scan file:", existing.sum(), "/", len(train_df))
if not existing.all():
    missing = train_df.loc[~existing, "image_id"].unique()
    print("Missing image_ids (first 10):", list(missing)[:10])

Train: rows with existing scan file: 1986 / 1986


In [14]:
train_df["image_path"] = train_df["image_id"].apply(lambda x: image_id_to_path(x))

In [15]:
train_df.head(10)

,id,image_id,object_id,patient_id,series_name,study_id,segmentMap,bbox,label,label_id,image_path
0,30930.000000-Recon 2 LOW DOSE THROUGH LUNGS-19...,30930.000000-Recon 2 LOW DOSE THROUGH LUNGS-19490,1,LIDC-IDRI-0939,1.3.6.1.4.1.14519.5.2.1.6279.6001.313835996725...,30930.000000-Recon 2 LOW DOSE THROUGH LUNGS-19490,"{'1': None, 'Calcification': ['Absent'], 'Inte...","[423, 258, 190, 425, 264, 193]",Moderately Unlikely,1,/kaggle/input/datasets/vladimirsydor/lidc-clas...
1,30930.000000-Recon 2 LOW DOSE THROUGH LUNGS-19...,30930.000000-Recon 2 LOW DOSE THROUGH LUNGS-19490,2,LIDC-IDRI-0939,1.3.6.1.4.1.14519.5.2.1.6279.6001.313835996725...,30930.000000-Recon 2 LOW DOSE THROUGH LUNGS-19490,"{'1': None, 'Calcification': ['Absent'], 'Inte...","[125, 217, 259, 132, 223, 263]",Moderately Unlikely,1,/kaggle/input/datasets/vladimirsydor/lidc-clas...
2,30930.000000-Recon 2 LOW DOSE THROUGH LUNGS-19...,30930.000000-Recon 2 LOW DOSE THROUGH LUNGS-19490,3,LIDC-IDRI-0939,1.3.6.1.4.1.14519.5.2.1.6279.6001.313835996725...,30930.000000-Recon 2 LOW DOSE THROUGH LUNGS-19490,"{'1': None, 'Calcification': ['Absent', 'Absen...","[111, 210, 367, 129, 236, 389]",Moderately Suspicious,2,/kaggle/input/datasets/vladimirsydor/lidc-clas...
3,30006.000000-NA-79830__1,30006.000000-NA-79830,1,LIDC-IDRI-0543,1.3.6.1.4.1.14519.5.2.1.6279.6001.792381786708...,30006.000000-NA-79830,"{'1': None, 'Calcification': ['Absent', 'Absen...","[423, 197, 390, 435, 214, 400]",Indeterminate,-1,/kaggle/input/datasets/vladimirsydor/lidc-clas...
4,30006.000000-NA-79830__2,30006.000000-NA-79830,2,LIDC-IDRI-0543,1.3.6.1.4.1.14519.5.2.1.6279.6001.792381786708...,30006.000000-NA-79830,"{'1': None, 'Calcification': ['Absent', 'Absen...","[399, 186, 411, 424, 212, 439]",Moderately Suspicious,2,/kaggle/input/datasets/vladimirsydor/lidc-clas...
5,3.000000-NA-26488__1,3.000000-NA-26488,1,LIDC-IDRI-0842,1.3.6.1.4.1.14519.5.2.1.6279.6001.245349763807...,3.000000-NA-26488,"{'1': None, 'Calcification': ['Absent', 'Absen...","[379, 216, 86, 383, 220, 87]",Indeterminate,-1,/kaggle/input/datasets/vladimirsydor/lidc-clas...
6,3.000000-NA-26488__2,3.000000-NA-26488,2,LIDC-IDRI-0842,1.3.6.1.4.1.14519.5.2.1.6279.6001.245349763807...,3.000000-NA-26488,"{'1': None, 'Calcification': ['Solid', 'Solid'...","[323, 362, 89, 329, 366, 90]",Highly Unlikely,0,/kaggle/input/datasets/vladimirsydor/lidc-clas...
7,1785.000000-NLST TLC VOL B30F-26096__1,1785.000000-NLST TLC VOL B30F-26096,1,LIDC-IDRI-0318,1.3.6.1.4.1.14519.5.2.1.6279.6001.439153572396...,1785.000000-NLST TLC VOL B30F-26096,"{'1': None, 'Calcification': ['Absent', 'Solid...","[45, 238, 225, 63, 256, 235]",Highly Unlikely,0,/kaggle/input/datasets/vladimirsydor/lidc-clas...
8,2.000000-NA-69862__1,2.000000-NA-69862,1,LIDC-IDRI-0752,1.3.6.1.4.1.14519.5.2.1.6279.6001.297988578825...,2.000000-NA-69862,"{'1': None, 'Calcification': ['Absent', 'Absen...","[400, 195, 127, 410, 205, 131]",Indeterminate,-1,/kaggle/input/datasets/vladimirsydor/lidc-clas...
9,2.000000-NA-69862__2,2.000000-NA-69862,2,LIDC-IDRI-0752,1.3.6.1.4.1.14519.5.2.1.6279.6001.297988578825...,2.000000-NA-69862,"{'1': None, 'Calcification': ['Absent', 'Absen...","[317, 291, 171, 334, 318, 180]",Moderately Suspicious,2,/kaggle/input/datasets/vladimirsydor/lidc-clas...


In [16]:
existing = train_df["image_path"].apply(lambda p: p.exists())
existing.value_counts()

image_path
True    1986
Name: count, dtype: int64

In [17]:
row = train_df.iloc[0]
path = Path(row["image_path"])

img = nib.load(str(path))
vol = img.get_fdata() # volume
shape = vol.shape
bbox = np.array(row["bbox"], dtype=int)  # [x0, y0, z0, x1, y1, z1]

print("Sample volume shape (x,y,z):", shape)
print("Sample bbox [x0,y0,z0,x1,y1,z1]:", bbox.tolist())
print("Dtype, value range:", vol.dtype, float(vol.min()), "<=>",  float(vol.max()))

Sample volume shape (x,y,z): (512, 512, 465)
Sample bbox [x0,y0,z0,x1,y1,z1]: [423, 258, 190, 425, 264, 193]
Dtype, value range: float64 -3024.0 <=> 1798.0


In [18]:
patients = train_df["patient_id"].nunique()
scans = train_df["image_id"].nunique()
nodules = len(train_df)

print("Unique patients:", patients)
print("Unique scans (image_id):", scans)
print("Total nodules (rows):", nodules)
print()
print("Nodules per patient (min, median, max):", train_df.groupby("patient_id").size().agg(["min", "median", "max"]).tolist())
print("Nodules per scan (min, median, max):", train_df.groupby("image_id").size().agg(["min", "median", "max"]).tolist())

Unique patients: 680
Unique scans (image_id): 680
Total nodules (rows): 1986

Nodules per patient (min, median, max): [1.0, 2.0, 21.0]
Nodules per scan (min, median, max): [1.0, 2.0, 21.0]


In [43]:
# Train vs test overlap: do we share patients or scans? (affects CV–LB correlation)
if test_df is not None and "patient_id" in test_df.columns:
    train_patients = set(train_df["patient_id"].unique())
    test_patients = set(test_df["patient_id"].unique())
    overlap_patients = train_patients & test_patients
    print("Patients in train:", len(train_patients))
    print("Patients in test:", len(test_patients))
    print("Overlap (same patient in both):", len(overlap_patients))
if test_df is not None and "image_id" in test_df.columns:
    train_scans = set(train_df["image_id"].unique())
    test_scans = set(test_df["image_id"].unique())
    overlap_scans = train_scans & test_scans
    print("\nScans in train:", len(train_scans))
    print("Scans in test:", len(test_scans))
    print("Overlap (same scan in both):", len(overlap_scans))
if test_df is None:
    print("Test file not found; run on Kaggle to check train/test overlap.")

Patients in train: 680
Patients in test: 143
Overlap (same patient in both): 0

Scans in train: 680
Scans in test: 143
Overlap (same scan in both): 0


In [19]:
# Label distribution (incl. Indeterminate) — needed for stratified splits
print("Label distribution (train):")
print(train_df["label"].value_counts().sort_index())
print()
print("label_id distribution:")
print(train_df["label_id"].value_counts().sort_index())
# For validation we typically exclude Indeterminate (label_id == -1) or treat separately
train_with_label = train_df[train_df["label_id"] >= 0]
print("\nRows with valid label (label_id >= 0):", len(train_with_label))
print("Class counts (valid only):", train_with_label["label_id"].value_counts().sort_index().to_dict())

Label distribution (train):
label
Highly Suspicious        131
Highly Unlikely          250
Indeterminate            918
Moderately Suspicious    269
Moderately Unlikely      418
Name: count, dtype: int64

label_id distribution:
label_id
-1    918
 0    250
 1    418
 2    269
 3    131
Name: count, dtype: int64

Rows with valid label (label_id >= 0): 1068
Class counts (valid only): {0: 250, 1: 418, 2: 269, 3: 131}


In [20]:
per_patient_labels = train_with_label.groupby("patient_id")["label_id"].nunique()
patients_single_class = (per_patient_labels == 1).sum()
patients_multi_class = (per_patient_labels > 1).sum()
print("Patients with only one class (nodules):", patients_single_class)
print("Patients with multiple classes:", patients_multi_class)

Patients with only one class (nodules): 339
Patients with multiple classes: 180


## Validation

- **Group by `patient_id`** so the same patient never appears in both train and val
- **Stratify by `label_id`** (on rows with `label_id >= 0`) so each fold keeps similar class distribution

In [21]:
train_valid = train_df[train_df["label_id"] >= 0].copy()
train_df["fold"] = -1

In [22]:
sgk = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [23]:
for fold_num, (train_idx, val_idx) in enumerate(sgk.split(train_valid, train_valid["label_id"], train_valid["patient_id"])):
    train_df.loc[train_valid.index[val_idx], "fold"] = fold_num

In [24]:
train_df[train_df["fold"] >= 0]["fold"].value_counts().sort_index()

fold
0    232
1    198
2    219
3    204
4    215
Name: count, dtype: int64

In [25]:
from __future__ import annotations

from pathlib import Path
from typing import Iterable, Optional, Tuple, Union, Sequence, List

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import pytorch_lightning as pl
import torchmetrics
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle
from IPython.display import HTML

from monai.data import CacheNTransDataset

from monai.transforms import (
    MapTransform,
)
from monai.config import KeysCollection
from monai.utils import ensure_tuple_rep



class CenterCropAroundBBoxd(MapTransform):
    """
    Crop a fixed-size 3D patch centered at the bbox center.
    Expects bbox in voxel coords: [x0, y0, z0, x1, y1, z1] (inclusive max is OK).

    Input dict must contain:
      - image key (e.g. "image") holding a tensor/ndarray with shape [C, X, Y, Z]
      - bbox_key (default "bbox") holding array-like length 6

    Output:
      - replaces image with cropped patch [C, crop_x, crop_y, crop_z]
      - also adds "bbox_in_crop" with bbox coords in crop space (clipped)
    """

    def __init__(
        self,
        keys: KeysCollection,
        bbox_key: str = "bbox",
        crop_size: Union[Sequence[int], int] = (64, 64, 64),
        pad_mode: str = "constant",
        pad_value: float = 0,  
        allow_missing_keys: bool = False,
    ):
        super().__init__(keys, allow_missing_keys)
        self.bbox_key = bbox_key
        self.crop_size = ensure_tuple_rep(crop_size, 3)
        self.pad_mode = pad_mode
        self.pad_value = pad_value

    @staticmethod
    def _clip_bbox(b: np.ndarray, shape_xyz: Tuple[int, int, int]) -> np.ndarray:
        sx, sy, sz = shape_xyz
        x0, y0, z0, x1, y1, z1 = b.tolist()
        x0, x1 = sorted((x0, x1))
        y0, y1 = sorted((y0, y1))
        z0, z1 = sorted((z0, z1))
        x0 = int(np.clip(x0, 0, sx - 1)); x1 = int(np.clip(x1, 0, sx - 1))
        y0 = int(np.clip(y0, 0, sy - 1)); y1 = int(np.clip(y1, 0, sy - 1))
        z0 = int(np.clip(z0, 0, sz - 1)); z1 = int(np.clip(z1, 0, sz - 1))
        return np.array([x0, y0, z0, x1, y1, z1], dtype=int)

    def __call__(self, data):
        d = dict(data)

        if self.bbox_key not in d:
            raise KeyError(f"Missing '{self.bbox_key}' in data dict. Provide bbox as [x0,y0,z0,x1,y1,z1].")

        bbox = np.asarray(d[self.bbox_key], dtype=int).reshape(-1)
        if bbox.shape[0] != 6:
            raise ValueError(f"bbox must have 6 values [x0,y0,z0,x1,y1,z1], got shape {bbox.shape}")

        for k in self.keys:
            img = d[k]
            # Expect [C, X, Y, Z]
            if img.ndim != 4:
                raise ValueError(f"Expected image '{k}' with shape [C,X,Y,Z], got {tuple(img.shape)}")

            C, X, Y, Z = img.shape
            bbox_c = self._clip_bbox(bbox, (X, Y, Z))
            x0, y0, z0, x1, y1, z1 = bbox_c.tolist()

            # center of bbox (voxel space)
            cx = (x0 + x1) / 2.0
            cy = (y0 + y1) / 2.0
            cz = (z0 + z1) / 2.0

            sx, sy, sz = self.crop_size
            hx, hy, hz = sx // 2, sy // 2, sz // 2

            # crop bounds in source volume (half-open)
            xs0 = int(np.round(cx)) - hx
            ys0 = int(np.round(cy)) - hy
            zs0 = int(np.round(cz)) - hz
            xs1 = xs0 + sx
            ys1 = ys0 + sy
            zs1 = zs0 + sz

            # intersection with image
            xi0, xi1 = max(xs0, 0), min(xs1, X)
            yi0, yi1 = max(ys0, 0), min(ys1, Y)
            zi0, zi1 = max(zs0, 0), min(zs1, Z)

            crop = img[:, xi0:xi1, yi0:yi1, zi0:zi1]

            # padding amounts
            pad_before_x = xi0 - xs0
            pad_before_y = yi0 - ys0
            pad_before_z = zi0 - zs0
            pad_after_x = xs1 - xi1
            pad_after_y = ys1 - yi1
            pad_after_z = zs1 - zi1

            # MONAI images are often torch tensors; use np.pad only if numpy
            is_torch = hasattr(crop, "detach") and hasattr(crop, "cpu")
            if is_torch:
                import torch
                pad = (pad_before_z, pad_after_z, pad_before_y, pad_after_y, pad_before_x, pad_after_x)  # reverse order
                crop = torch.nn.functional.pad(
                    crop,
                    pad=pad,
                    mode="constant" if self.pad_mode == "constant" else self.pad_mode,
                    value=self.pad_value,
                )
            else:
                crop = np.pad(
                    crop,
                    pad_width=((0, 0),
                               (pad_before_x, pad_after_x),
                               (pad_before_y, pad_after_y),
                               (pad_before_z, pad_after_z)),
                    mode=self.pad_mode,
                    constant_values=self.pad_value if self.pad_mode == "constant" else 0,
                )

            # bbox mapped into crop space (clipped)
            bx0 = int(np.clip(x0 - xs0, 0, sx - 1)); bx1 = int(np.clip(x1 - xs0, 0, sx - 1))
            by0 = int(np.clip(y0 - ys0, 0, sy - 1)); by1 = int(np.clip(y1 - ys0, 0, sy - 1))
            bz0 = int(np.clip(z0 - zs0, 0, sz - 1)); bz1 = int(np.clip(z1 - zs0, 0, sz - 1))

            d[k] = crop
            d["bbox_in_crop"] = np.array([bx0, by0, bz0, bx1, by1, bz1], dtype=int)
            d["crop_origin_xyz"] = np.array([xs0, ys0, zs0], dtype=int)  # useful for debugging

        return d


class LitCNNModule(pl.LightningModule):
    def __init__(
        self, 
        model_class,
        model_kwargs,
        n_classes,
        initial_lr=1e-3, 
        final_lr=1e-5, 
        T_max=10,
        class_weights=None
    ):
        super().__init__()
        self.save_hyperparameters()
        self.model = model_class(**model_kwargs)
        self.criterion = nn.CrossEntropyLoss(
            weight=class_weights
        )
        self.train_metrics = torchmetrics.MetricCollection(metrics=[
            torchmetrics.F1Score(task="multiclass", num_classes=n_classes, num_labels=n_classes, average="macro"),
            torchmetrics.Recall(task="multiclass", num_classes=n_classes, num_labels=n_classes, average="macro"),
            torchmetrics.Precision(task="multiclass", num_classes=n_classes, num_labels=n_classes, average="macro")
        ], compute_groups=False)  
        self.val_metrics = torchmetrics.MetricCollection(metrics=[
            torchmetrics.F1Score(task="multiclass", num_classes=n_classes, num_labels=n_classes, average="macro"),
            torchmetrics.Recall(task="multiclass", num_classes=n_classes, num_labels=n_classes, average="macro"),
            torchmetrics.Precision(task="multiclass", num_classes=n_classes, num_labels=n_classes, average="macro")
        ], compute_groups=False)
        self.initial_lr = initial_lr
        self.final_lr = final_lr
        self.T_max = T_max

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch["image"], batch["label"]
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=-1)
        self.train_metrics.update(preds, y)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def on_train_epoch_end(self):
        self.log_dict(
            {"train_" + k:v for k,v in self.train_metrics.compute().items()},
            on_step=False,
            on_epoch=True,
            prog_bar=True,
            sync_dist=True,
        )
        self.train_metrics.reset()

    def validation_step(self, batch, batch_idx):
        x, y = batch["image"], batch["label"]
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = logits.argmax(dim=-1)
        self.val_metrics.update(preds, y)
        self.log("val_loss", loss, on_epoch=True, prog_bar=True)

    def on_validation_epoch_end(self):
        self.log_dict(
            {"val_" + k:v for k,v in self.val_metrics.compute().items()},
            on_step=False,
            on_epoch=True,
            prog_bar=True,
            sync_dist=True,
        )
        self.val_metrics.reset()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(
            self.parameters(), 
            lr=self.hparams.initial_lr
        )
        scheduler = {
            "scheduler": torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer, 
                T_max=self.hparams.T_max,
                eta_min=self.hparams.final_lr
            ),
            "interval": "step"
        }
        return (
            [optimizer],
            [scheduler]
        )


class LitImageDataModule(pl.LightningDataModule):
    def __init__(
        self, 
        df, 
        n_static_transforms,
        cache_dir,
        path_column="image_path",
        bbox_column="bbox",
        label_column="label_id",
        batch_size=32, 
        num_workers=4, 
        fold=0, 
        val_tfms=None, 
        train_tfms=None
    ):
        super().__init__()
        self.df = df
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.fold = fold
        self.n_static_transforms = n_static_transforms
        self.cache_dir = cache_dir
        self.path_column = path_column
        self.bbox_column = bbox_column
        self.label_column = label_column

        self.train_tfms = train_tfms
        self.val_tfms = val_tfms
        

    def setup(self, stage=None):
        train_df = self.df[self.df["fold"] != self.fold].reset_index(drop=True)
        val_df = self.df[self.df["fold"] == self.fold].reset_index(drop=True)

        train_data = [
            {
                "image": img,
                "bbox": bbox,
                "label": label
            } for img, bbox, label in zip(train_df[self.path_column], train_df[self.bbox_column], train_df[self.label_column])
        ]
        val_data = [
            {
                "image": img,
                "bbox": bbox,
                "label": label
            } for img, bbox, label in zip(val_df[self.path_column], val_df[self.bbox_column], val_df[self.label_column])
        ]

        self.train_ds = CacheNTransDataset(train_data, self.train_tfms, cache_n_trans=self.n_static_transforms, cache_dir=self.cache_dir)
        self.val_ds = CacheNTransDataset(val_data, self.val_tfms, cache_n_trans=self.n_static_transforms, cache_dir=self.cache_dir)

    def train_dataloader(self):
        return torch.utils.data.DataLoader(
            self.train_ds, batch_size=self.batch_size, 
            shuffle=True, pin_memory=torch.cuda.is_available(),
            num_workers=self.num_workers
        )

    def val_dataloader(self):
        return torch.utils.data.DataLoader(
            self.val_ds, batch_size=self.batch_size, 
            shuffle=False, pin_memory=torch.cuda.is_available(),
            num_workers=self.num_workers
        )

In [26]:
CROP_SIZE = (64, 64, 64)

train_transforms = monai.transforms.Compose([
    monai.transforms.LoadImaged(keys=["image"], dtype="float32"),
    monai.transforms.EnsureChannelFirstd(keys=["image"]),
    CenterCropAroundBBoxd(
        keys=["image"], bbox_key="bbox", crop_size=CROP_SIZE,
        pad_mode="constant", pad_value=-1024.0,
    ),
    monai.transforms.ScaleIntensityRanged(
        keys=["image"], a_min=-1024, a_max=700.0, b_min=0.0, b_max=1.0, clip=True,
    ),
    monai.transforms.RandFlipd(keys=["image"], prob=0.5, spatial_axis=0),
    monai.transforms.RandFlipd(keys=["image"], prob=0.5, spatial_axis=1),
    monai.transforms.RandFlipd(keys=["image"], prob=0.5, spatial_axis=2),
    monai.transforms.RandRotate90d(keys=["image"], prob=0.4, max_k=3, spatial_axes=(0, 1)),
    monai.transforms.RandZoomd(keys=["image"], prob=0.2, min_zoom=0.9, max_zoom=1.1, mode="trilinear", align_corners=False),
    monai.transforms.RandGaussianNoised(keys=["image"], prob=0.15, mean=0.0, std=0.02),
    monai.transforms.RandGaussianSmoothd(keys=["image"], prob=0.15, sigma_x=(0.5, 1.0), sigma_y=(0.5, 1.0), sigma_z=(0.5, 1.0)),
])

val_transforms = monai.transforms.Compose([
    monai.transforms.LoadImaged(keys=["image"], dtype="float32"),
    monai.transforms.EnsureChannelFirstd(keys=["image"]),
    CenterCropAroundBBoxd(
        keys=["image"], bbox_key="bbox", crop_size=CROP_SIZE,
        pad_mode="constant", pad_value=-1024.0,
    ),
    monai.transforms.ScaleIntensityRanged(
        keys=["image"], a_min=-1024, a_max=700.0, b_min=0.0, b_max=1.0, clip=True,
    ),
])

In [27]:
# Class weights for imbalanced classes (optional)
df_cv = train_df[train_df["fold"] >= 0]
labels_np = df_cv["label_id"].values
classes = np.unique(labels_np)
weights = compute_class_weight("balanced", classes=classes, y=labels_np)
class_weights = torch.tensor(weights, dtype=torch.float32)
print("Class weights:", class_weights.tolist())

Class weights: [1.0679999589920044, 0.6387559771537781, 0.9925650358200073, 2.038167953491211]


In [28]:
# 2D classifier: 3D volume -> center slice -> 2D CNN (EfficientNet)
class Timm2DClassifier(nn.Module):
    def __init__(self, timm_config):
        super().__init__()
        self.cnn2d = timm.create_model(**timm_config)

    def forward(self, volume):
        z_mid = volume.shape[-1] // 2
        slice_2d = volume[..., z_mid]
        return self.cnn2d(slice_2d)

In [29]:
# Training config and W&B
RUN_NAME = "effnet2d_fold0"
WANDB_PROJECT = "lidc-nodule-classification"  # change to your project
FOLD = 0
N_EPOCHS = 20
BATCH_SIZE = 8
CACHE_DIR = "temp_cache"

df_for_cv = train_df[train_df["fold"] >= 0].reset_index(drop=True)
n_train = (df_for_cv["fold"] != FOLD).sum()
n_steps_per_epoch = max(1, n_train // BATCH_SIZE)
T_max = N_EPOCHS * n_steps_per_epoch

wandb_logger = WandbLogger(
    project=WANDB_PROJECT,
    name=RUN_NAME,
    save_dir=".",
)

In [37]:
# Train all 5 folds (2D EfficientNet)
fold_scores = []
N_FOLDS = 5

for FOLD in range(N_FOLDS):
    RUN_NAME = f"effnet2d_fold{FOLD}"
    n_train = (df_for_cv["fold"] != FOLD).sum()
    n_steps_per_epoch = max(1, n_train // BATCH_SIZE)
    T_max = N_EPOCHS * n_steps_per_epoch

    wandb_logger = WandbLogger(
        project=WANDB_PROJECT,
        name=RUN_NAME,
        save_dir=".",
    )

    datamodule = LitImageDataModule(
        df=df_for_cv,
        n_static_transforms=3,
        cache_dir=CACHE_DIR,
        path_column="image_path",
        bbox_column="bbox",
        label_column="label_id",
        batch_size=BATCH_SIZE,
        num_workers=4,
        fold=FOLD,
        train_tfms=train_transforms,
        val_tfms=val_transforms,
    )

    lit_model = LitCNNModule(
        model_class=Timm2DClassifier,
        model_kwargs=dict(
            timm_config={
                "model_name": "tf_efficientnet_b3.in1k",
                "num_classes": 4,
                "pretrained": True,
                "in_chans": 1,
            }
        ),
        n_classes=4,
        initial_lr=1e-3,
        final_lr=1e-5,
        T_max=T_max,
        class_weights=class_weights,
    )

    checkpoint_cb = ModelCheckpoint(
        monitor="val_MulticlassF1Score",
        mode="max",
        save_top_k=1,
        dirpath="checkpoints",
        filename=RUN_NAME + "-{epoch:02d}-{val_MulticlassF1Score:.4f}",
    )
    lr_monitor = LearningRateMonitor(logging_interval="step")

    trainer = pl.Trainer(
        max_epochs=N_EPOCHS,
        logger=wandb_logger,
        callbacks=[checkpoint_cb, lr_monitor],
        accelerator="auto",
        devices=1,
        log_every_n_steps=10,
    )

    datamodule.setup()
    trainer.fit(lit_model, datamodule=datamodule)
    best_f1 = checkpoint_cb.best_model_score.item() if checkpoint_cb.best_model_score is not None else float("nan")
    fold_scores.append(best_f1)
    wandb.finish()

print("Fold Macro F1 scores:", fold_scores)
print("Mean:", np.mean(fold_scores), "± Std:", np.std(fold_scores))

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model         │ Timm2DClassifier │ 10.7 M │ train │     0 │
│ 1 │ criterion     │ CrossEntropyLoss │      0 │ train │     0 │
│ 2 │ train_metrics │ MetricCollection │      0 │ train │     0 │
│ 3 │ val_metrics   │ MetricCollection │      0 │ train │     0 │
└───┴───────────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 10.7 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 10.7 M                                                                                               
Total estimated model params size (MB): 42                                                                         
Modules in train mode: 543                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

Trainable params: 10.7 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 10.7 M                                                                                               
Total estimated model params size (MB): 42                                                                         
Modules in train mode: 543                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=20` reached.


epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▇▇▇▇▇▇▇▇▇█████
lr-Adam,██████████▇▇▇▇▆▆▆▅▅▅▅▅▅▅▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁
train_MulticlassF1Score,▁▂▃▄▅▅▅▅▅▆▆▆▇▇▇███▇█
train_MulticlassPrecision,▁▂▃▄▄▅▅▅▅▆▆▆█▇▇███▇█
train_MulticlassRecall,▁▂▃▄▅▆▅▅▅▆▇▆▇▇▇███▇█
train_loss_epoch,█▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
train_loss_step,█▆▄▂▃▂▂▂▂▂▃▂▂▁▃▂▁▂▂▁▂▂▂▁▁▂▂▂▂▁▁▂▂▁▁▂▁▂▁▂
trainer/global_step,▁▁▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▆▇▇▇▇▇▇█
val_MulticlassF1Score,▁▄▅▅▅▅▃▇▆▆▇▇▇▇████▇█
val_MulticlassPrecision,▁▄▅▆▅▅▄█▆▆█▇▇███▇█▇█
+2,...


Fold Macro F1 scores: [0.6162043213844299, 0.6138521432876587, 0.6277377605438232, 0.654852032661438, 0.6627426147460938]
Mean: 0.6350777745246887 ± Std: 0.020084696310484228


In [39]:
print("Fold Macro F1 scores:", fold_scores)
print("Mean:", np.mean(fold_scores), "± Std:", np.std(fold_scores))

Fold Macro F1 scores: [0.6162043213844299, 0.6138521432876587, 0.6277377605438232, 0.654852032661438, 0.6627426147460938]
Mean: 0.6350777745246887 ± Std: 0.020084696310484228


# Test inf

In [49]:
test_df

,id,image_id,object_id,patient_id,series_name,study_id,bbox,image_path
0,3000025.000000-NA-58100__1,3000025.000000-NA-58100,1,LIDC-IDRI-0634,1.3.6.1.4.1.14519.5.2.1.6279.6001.162901839201...,3000025.000000-NA-58100,[127 119 18 133 124 20],/kaggle/input/datasets/vladimirsydor/lidc-clas...
1,4.000000-Recon 3 C-A-P-68794__1,4.000000-Recon 3 C-A-P-68794,1,LIDC-IDRI-0243,1.3.6.1.4.1.14519.5.2.1.6279.6001.939216568327...,4.000000-Recon 3 C-A-P-68794,[168 292 55 186 313 57],/kaggle/input/datasets/vladimirsydor/lidc-clas...
2,4.000000-Recon 3 C-A-P-68794__2,4.000000-Recon 3 C-A-P-68794,2,LIDC-IDRI-0243,1.3.6.1.4.1.14519.5.2.1.6279.6001.939216568327...,4.000000-Recon 3 C-A-P-68794,[192 116 69 198 119 71],/kaggle/input/datasets/vladimirsydor/lidc-clas...
3,30906.000000-Recon 2 ACRIN LARGE-35112__4,30906.000000-Recon 2 ACRIN LARGE-35112,4,LIDC-IDRI-0469,1.3.6.1.4.1.14519.5.2.1.6279.6001.258220324170...,30906.000000-Recon 2 ACRIN LARGE-35112,[ 75 215 376 89 226 390],/kaggle/input/datasets/vladimirsydor/lidc-clas...
4,30906.000000-Recon 2 ACRIN LARGE-35112__5,30906.000000-Recon 2 ACRIN LARGE-35112,5,LIDC-IDRI-0469,1.3.6.1.4.1.14519.5.2.1.6279.6001.258220324170...,30906.000000-Recon 2 ACRIN LARGE-35112,[311 168 401 318 174 404],/kaggle/input/datasets/vladimirsydor/lidc-clas...
...,...,...,...,...,...,...,...,...
275,4155.000000-NLST TLC VOL B30F-21227__5,4155.000000-NLST TLC VOL B30F-21227,5,LIDC-IDRI-0487,1.3.6.1.4.1.14519.5.2.1.6279.6001.300270516469...,4155.000000-NLST TLC VOL B30F-21227,[325 74 176 343 90 179],/kaggle/input/datasets/vladimirsydor/lidc-clas...
276,4155.000000-NLST TLC VOL B30F-21227__6,4155.000000-NLST TLC VOL B30F-21227,6,LIDC-IDRI-0487,1.3.6.1.4.1.14519.5.2.1.6279.6001.300270516469...,4155.000000-NLST TLC VOL B30F-21227,[116 359 198 129 371 203],/kaggle/input/datasets/vladimirsydor/lidc-clas...
277,5482.000000-NA-36928__1,5482.000000-NA-36928,1,LIDC-IDRI-0354,1.3.6.1.4.1.14519.5.2.1.6279.6001.112020141051...,5482.000000-NA-36928,[ 74 166 59 88 179 61],/kaggle/input/datasets/vladimirsydor/lidc-clas...
278,3000041.000000-NA-15660__1,3000041.000000-NA-15660,1,LIDC-IDRI-0732,1.3.6.1.4.1.14519.5.2.1.6279.6001.244681063194...,3000041.000000-NA-15660,[436 276 66 444 284 67],/kaggle/input/datasets/vladimirsydor/lidc-clas...


In [51]:
if test_df is not None:
    if "image_id" in test_df.columns:
        test_df = test_df.copy()
        test_df["image_path"] = test_df["image_id"].apply(image_id_to_path)
    if "bbox" in test_df.columns and len(test_df) > 0 and isinstance(test_df["bbox"].iloc[0], str):
        def parse_bbox(s):
            s = str(s).strip()
            if s.startswith("["): s = s[1:]
            if s.endswith("]"): s = s[:-1]
            return [int(x) for x in s.split()]
        test_df["bbox"] = test_df["bbox"].apply(parse_bbox)

In [52]:
# Dataset that applies val_transforms and returns (image tensor, id)
class TestNoduleDataset(torch.utils.data.Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        d = {"image": str(row["image_path"]), "bbox": np.array(row["bbox"], dtype=np.int64)}
        d = self.transform(d)
        return d["image"], row["id"]

test_ds = TestNoduleDataset(test_df, val_transforms)
test_loader = torch.utils.data.DataLoader(
    test_ds, batch_size=8, shuffle=False, num_workers=0, pin_memory=False
)

In [67]:
ckpt_dir = Path("checkpoints")
ckpt_files = sorted(ckpt_dir.glob("effnet2d_fold*-*.ckpt"))
if not ckpt_files:
    raise FileNotFoundError("No checkpoints found in checkpoints/. Train first or set ckpt path.")
ckpt_path = str(ckpt_files[1])  # use first fold; change index for another fold
print("Loading:", ckpt_path)

Loading: checkpoints/effnet2d_fold0-epoch=15-val_MulticlassF1Score=0.6856.ckpt


In [69]:
model = LitCNNModule.load_from_checkpoint(
    ckpt_path,
    model_class=Timm2DClassifier,
    model_kwargs=dict(
        timm_config={
            "model_name": "tf_efficientnet_b3.in1k",
            "num_classes": 4,
            "pretrained": False,
            "in_chans": 1,
        }
    ),
    n_classes=4,
    weights_only=False,  # required for checkpoints with numpy/torchmetrics (PyTorch 2.6+)
)

In [70]:
# Run inference and collect (id, predicted)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
ids_list = []
preds_list = []
with torch.no_grad():
    for images, ids in test_loader:
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim=1).cpu().numpy()
        ids_batch = ids.tolist() if hasattr(ids, "tolist") else list(ids)
        ids_list.extend(ids_batch)
        preds_list.extend(preds.tolist())

submission = pd.DataFrame({"id": ids_list, "predicted": preds_list})
submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)
print(submission.head(10))

Saved: submission.csv
                                          id  predicted
0                 3000025.000000-NA-58100__1          0
1            4.000000-Recon 3 C-A-P-68794__1          3
2            4.000000-Recon 3 C-A-P-68794__2          0
3  30906.000000-Recon 2 ACRIN LARGE-35112__4          1
4  30906.000000-Recon 2 ACRIN LARGE-35112__5          0
5     1624.000000-NLST TLC VOL B30F-61989__2          0
6                   30889.000000-NA-98603__1          0
7                   30889.000000-NA-98603__3          1
8                   30889.000000-NA-98603__4          2
9            2.000000-BOTTOM TO TOP-22938__1          2
